# 🚀 Projeto Final – Simulação de Sistema Paralelo em Tempo Real
Disciplina: Computação Paralela

Este notebook implementa um sistema de processamento em fluxo contínuo (streaming), explorando concorrência, paralelismo e pipeline.

---

## 🎯 Objetivo
Comparar diferentes estratégias de execução:
- Sequencial
- Concorrente (Threads)
- Paralelo (Processos)
- Pipeline (Produtor-Consumidor)

E analisar desempenho, gargalos e uso de hardware.

In [1]:
import time
import random
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from queue import Queue
from threading import Thread
import multiprocessing

## 🔹 Gerador de Eventos (Streaming)
Simula eventos chegando em tempo real.

In [2]:
def gerar_eventos(n):
    for _ in range(n):
        evento = {
            'user_id': random.randint(1, 100),
            'item_id': random.randint(1, 1000),
            'timestamp': time.time()
        }
        yield evento

## 🔹 Função de Processamento
Simula carga computacional.

In [3]:
#def processar_evento(evento):
#    time.sleep(0.01)
#    return evento['user_id'] * evento['item_id']


def processar_evento(evento):
    total = 0
    limite = 300000
    base = evento["user_id"] * evento["item_id"]
    for i in range(limite):
        total += (base + i) % 97
    return total    

## 🔹 Execução Sequencial

In [4]:
start = time.time()

for e in gerar_eventos(100):
    processar_evento(e)

end = time.time()
tempo_sequencial = end - start
print('Tempo sequencial:', tempo_sequencial)

Tempo sequencial: 1.0945696830749512


## 🔹 Execução Concorrente (Threads)

In [5]:
start = time.time()

with ThreadPoolExecutor(max_workers=4) as executor:
    list(executor.map(processar_evento, gerar_eventos(100)))

end = time.time()
tempo_thread = end - start
print('Tempo concorrente:', tempo_thread)

Tempo concorrente: 1.1672520637512207


## 🔹 Execução Paralela (Processos)

In [6]:
start = time.time()

with ProcessPoolExecutor(max_workers=4) as executor:
    list(executor.map(processar_evento, gerar_eventos(100)))

end = time.time()
tempo_process = end - start
print('Tempo paralelo:', tempo_process)

Tempo paralelo: 0.38458824157714844


## 🔹 Pipeline Produtor-Consumidor

In [7]:
fila_entrada = Queue()
fila_saida = Queue()

def produtor():
    for e in gerar_eventos(100):
        fila_entrada.put(e)

def consumidor():
    while True:
        e = fila_entrada.get()
        resultado = processar_evento(e)
        fila_saida.put(resultado)
        fila_entrada.task_done()

start = time.time()

Thread(target=produtor).start()

for _ in range(4):
    Thread(target=consumidor, daemon=True).start()

fila_entrada.join()

end = time.time()
tempo_pipeline = end - start
print('Tempo pipeline:', tempo_pipeline)

Tempo pipeline: 1.1751844882965088


## 🔹 Diagnóstico de Hardware

In [8]:
print('CPUs disponíveis:', multiprocessing.cpu_count())

CPUs disponíveis: 16


## 🔹 Análise de Speedup

In [9]:
print('Speedup Threads:', tempo_sequencial / tempo_thread)
print('Speedup Processos:', tempo_sequencial / tempo_process)
print('Speedup Pipeline:', tempo_sequencial / tempo_pipeline)

Speedup Threads: 0.9377320606804594
Speedup Processos: 2.8460820294095766
Speedup Pipeline: 0.9314024257260126


## 🧠 Reflexão Final
- Qual abordagem foi mais rápida?
- Onde houve gargalo?
- Threads ou processos foram melhores?
- O pipeline melhorou throughput?
- Como otimizar ainda mais?